In [8]:
pip install xlsxwriter 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 1. Check for Images with different SORTIE, DATE, FLIGHTNUMBER within a Folder

In [1]:
import os
import pandas as pd
from datetime import datetime
from collections import defaultdict
import re
import time
from concurrent.futures import ThreadPoolExecutor
import warnings
warnings.filterwarnings('ignore')

# Path configuration
folders_path = r"W:"
output_path = r"G:\0 BIPLOV\TASKS\FELIX"

# Get current date for filename
current_date = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create output filename
output_filename = f"{current_date}_SORTIE_Date_Flight_Validation.xlsx"
output_filepath = os.path.join(output_path, output_filename)

# Cache dictionaries for faster processing
folder_cache = {}
image_cache = {}

# Compile regex patterns for better performance
# More flexible date patterns to handle different formats
date_patterns = [
    re.compile(r'(\d{2})[-_](\d{2})[-_](\d{4})'),  # dd_mm_yyyy or dd-mm-yyyy
    re.compile(r'(\d{4})[-_](\d{2})[-_](\d{2})'),  # yyyy_mm_dd or yyyy-mm-dd
    re.compile(r'(\d{4})(\d{2})(\d{2})'),          # yyyymmdd
    re.compile(r'(\d{2})(\d{2})(\d{4})'),          # ddmmyyyy
]

suffix_pattern = re.compile(r'(_modified|_modifiziert|\.tif|\.jpg|\.png|\.bmp|\.tiff)', re.IGNORECASE)

def extract_date_components(date_str):
    """Extract year, month, day from date string in various formats"""
    if not date_str or not isinstance(date_str, str):
        return None, None, None
    
    # Clean the string
    clean_str = date_str.strip().replace(' ', '')
    
    for pattern in date_patterns:
        match = pattern.search(clean_str)
        if match:
            groups = match.groups()
            
            if len(groups) == 3:
                # Pattern 1: dd_mm_yyyy or dd-mm-yyyy
                if pattern == date_patterns[0]:
                    day, month, year = groups
                    return year, month, day
                
                # Pattern 2: yyyy_mm_dd or yyyy-mm-dd
                elif pattern == date_patterns[1]:
                    year, month, day = groups
                    return year, month, day
                
                # Pattern 3: yyyymmdd
                elif pattern == date_patterns[2]:
                    year, month, day = groups[0], groups[1], groups[2]
                    return year, month, day
                
                # Pattern 4: ddmmyyyy
                elif pattern == date_patterns[3]:
                    day, month, year = groups[0], groups[1], groups[2]
                    return year, month, day
    
    # If no pattern matches, try to parse as is
    try:
        # Try common separators
        for sep in ['-', '_', '.', '/']:
            if sep in clean_str:
                parts = clean_str.split(sep)
                if len(parts) == 3:
                    # Determine format by checking part lengths
                    if len(parts[0]) == 4:  # yyyy-mm-dd
                        year, month, day = parts[0], parts[1], parts[2]
                        return year, month, day
                    elif len(parts[2]) == 4:  # dd-mm-yyyy
                        day, month, year = parts[0], parts[1], parts[2]
                        return year, month, day
    except:
        pass
    
    return None, None, None

def dates_match(date1, date2):
    """Compare two dates by their components (year, month, day)"""
    if not date1 or not date2:
        return False
    
    year1, month1, day1 = extract_date_components(date1)
    year2, month2, day2 = extract_date_components(date2)
    
    # If we couldn't extract components from either date
    if None in (year1, month1, day1) or None in (year2, month2, day2):
        # Fallback to string comparison
        return str(date1) == str(date2)
    
    # Compare components (ensure 2-digit months/days)
    month1 = month1.zfill(2)
    month2 = month2.zfill(2)
    day1 = day1.zfill(2)
    day2 = day2.zfill(2)
    
    return year1 == year2 and month1 == month2 and day1 == day2

def normalize_date(date_str):
    """Normalize date to yyyy-mm-dd format for display"""
    if not date_str:
        return ""
    
    year, month, day = extract_date_components(date_str)
    if year and month and day:
        return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
    return date_str

def normalize_flight_number(flight_str):
    """Normalize flight number for comparison (remove leading zeros, spaces, etc.)"""
    if not flight_str:
        return ""
    
    # Remove any non-alphanumeric characters except letters and digits
    clean = re.sub(r'[^a-zA-Z0-9]', '', str(flight_str).strip())
    
    # Remove leading zeros if it's purely numeric
    if clean.isdigit():
        return clean.lstrip('0') or '0'
    
    return clean

def flights_match(flight1, flight2):
    """Compare two flight numbers with normalization"""
    if not flight1 or not flight2:
        return False
    
    norm1 = normalize_flight_number(flight1)
    norm2 = normalize_flight_number(flight2)
    
    return norm1 == norm2

def extract_folder_info(folder_name):
    """Extract SORTIE, Flight, and other information from folder name"""
    if folder_name in folder_cache:
        return folder_cache[folder_name]
    
    if not folder_name or '#' not in folder_name:
        folder_cache[folder_name] = None
        return None
    
    parts = folder_name.split('#')
    
    if len(parts) >= 3:
        folder_info = {
            'folder_name': folder_name,
            'folder_date': parts[0],  # Raw date from folder name
            'flight_number': parts[1].strip(),  # Flight number from folder
            'sortie': parts[2].strip(),  # Sortie from folder
            'full_path': os.path.join(folders_path, folder_name),
            'normalized_date': normalize_date(parts[0]),  # Normalized date for display
            'normalized_flight': normalize_flight_number(parts[1])  # Normalized flight
        }
        folder_cache[folder_name] = folder_info
        return folder_info
    
    folder_cache[folder_name] = None
    return None

def extract_image_info(image_name, folder_info):
    """Extract SORTIE, Flight, and Date information from image name"""
    cache_key = f"{image_name}_{folder_info['folder_name'] if folder_info else 'None'}"
    if cache_key in image_cache:
        return image_cache[cache_key]
    
    if not image_name or '#' not in image_name:
        image_cache[cache_key] = None
        return None
    
    # Remove file extensions and modifications
    clean_image_name = re.sub(suffix_pattern, '', image_name)
    parts = clean_image_name.split('#')
    
    if len(parts) >= 5:
        image_sortie = parts[2].strip() if len(parts) > 2 else ""
        image_flight = parts[1].strip() if len(parts) > 1 else ""
        
        # Extract date from the 5th part
        date_part = parts[4] if len(parts) > 4 else ""
        raw_image_date = date_part.strip()
        
        image_info = {
            'image_name': image_name,
            'image_sortie': image_sortie,
            'image_flight': image_flight,
            'raw_image_date': raw_image_date,  # Keep raw date
            'normalized_image_date': normalize_date(raw_image_date),  # Normalized for display
            'image_number': parts[0].strip() if len(parts) > 0 else "",
            'folder_sortie': folder_info['sortie'] if folder_info else "",
            'folder_flight': folder_info['flight_number'] if folder_info else "",
            'folder_date': folder_info['folder_date'] if folder_info else "",  # Raw folder date
            'normalized_folder_date': folder_info['normalized_date'] if folder_info else "",  # Normalized
            'normalized_folder_flight': folder_info['normalized_flight'] if folder_info else "",
            'folder_name': folder_info['folder_name'] if folder_info else ""
        }
        
        # Add validation flags using component-based comparison
        image_info['sortie_match'] = image_info['image_sortie'] == image_info['folder_sortie']
        image_info['date_match'] = dates_match(image_info['raw_image_date'], image_info['folder_date'])
        image_info['flight_match'] = flights_match(image_info['image_flight'], image_info['folder_flight'])
        
        image_cache[cache_key] = image_info
        return image_info
    
    image_cache[cache_key] = None
    return None

def process_folder_images(folder_path, folder_info):
    """Process all images in a folder and its subfolders"""
    mismatched_sorties = []
    mismatched_dates = []
    mismatched_flights = []
    
    try:
        # Walk through all subdirectories
        for root, dirs, files in os.walk(folder_path):
            # Process each image file
            for file in files:
                # Check for image extensions (case insensitive)
                if file.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png', '.bmp')):
                    image_info = extract_image_info(file, folder_info)
                    if image_info:
                        if not image_info['sortie_match']:
                            mismatched_sorties.append(image_info.copy())
                        if not image_info['date_match']:
                            mismatched_dates.append(image_info.copy())
                        if not image_info['flight_match']:
                            mismatched_flights.append(image_info.copy())
    except (PermissionError, FileNotFoundError) as e:
        print(f"Warning: Could not access {folder_path}: {e}")
    
    return mismatched_sorties, mismatched_dates, mismatched_flights

def process_all_folders():
    """Main function to process all folders and their images"""
    print("Scanning folders and images...")
    start_time = time.time()
    
    all_mismatched_sorties = []
    all_mismatched_dates = []
    all_mismatched_flights = []
    folder_stats = {'total_folders': 0, 'total_images': 0, 'processed_folders': 0}
    
    try:
        all_items = os.listdir(folders_path)
        folders = [item for item in all_items if os.path.isdir(os.path.join(folders_path, item))]
        folder_stats['total_folders'] = len(folders)
        
        print(f"Found {len(folders)} folders to process")
        
        # Process folders with progress indicator
        for i, folder in enumerate(folders):
            folder_info = extract_folder_info(folder)
            if folder_info:
                folder_path = folder_info['full_path']
                mismatched_sorties, mismatched_dates, mismatched_flights = process_folder_images(folder_path, folder_info)
                all_mismatched_sorties.extend(mismatched_sorties)
                all_mismatched_dates.extend(mismatched_dates)
                all_mismatched_flights.extend(mismatched_flights)
                
                folder_stats['total_images'] += len(mismatched_sorties) + len(mismatched_dates) + len(mismatched_flights)
                folder_stats['processed_folders'] += 1
            
            # Progress indicator
            if (i + 1) % 10 == 0 or (i + 1) == len(folders):
                elapsed = time.time() - start_time
                total_issues = len(all_mismatched_sorties) + len(all_mismatched_dates) + len(all_mismatched_flights)
                print(f"Progress: {i + 1}/{len(folders)} folders, "
                      f"{total_issues} issues found, "
                      f"{elapsed:.1f}s elapsed")
    
    except Exception as e:
        print(f"Error during processing: {e}")
    
    elapsed_time = time.time() - start_time
    print(f"\nProcessing completed in {elapsed_time:.2f} seconds")
    print(f"Processed {folder_stats['processed_folders']} folders")
    print(f"Found {len(all_mismatched_sorties)} SORTIE mismatches")
    print(f"Found {len(all_mismatched_dates)} Date mismatches")
    print(f"Found {len(all_mismatched_flights)} Flight mismatches")
    
    return all_mismatched_sorties, all_mismatched_dates, all_mismatched_flights

def create_dataframes(mismatched_sorties, mismatched_dates, mismatched_flights):
    """Create DataFrames from mismatch data"""
    
    # Create SORTIE mismatch DataFrame
    if mismatched_sorties:
        sortie_df = pd.DataFrame(mismatched_sorties)
        sortie_df = sortie_df[[
            'folder_name', 'folder_sortie', 'image_name', 'image_sortie',
            'image_number', 'image_flight', 'normalized_folder_date', 
            'normalized_image_date', 'raw_image_date', 'folder_date'
        ]]
        
        # Add discrepancy column
        sortie_df['discrepancy'] = sortie_df.apply(
            lambda x: f"Folder SORTIE: {x['folder_sortie']} ≠ Image SORTIE: {x['image_sortie']}", 
            axis=1
        )
    else:
        sortie_df = pd.DataFrame(columns=[
            'folder_name', 'folder_sortie', 'image_name', 'image_sortie',
            'image_number', 'image_flight', 'normalized_folder_date', 
            'normalized_image_date', 'raw_image_date', 'folder_date', 'discrepancy'
        ])
    
    # Create Date mismatch DataFrame
    if mismatched_dates:
        date_df = pd.DataFrame(mismatched_dates)
        date_df = date_df[[
            'folder_name', 'normalized_folder_date', 'image_name', 'normalized_image_date',
            'image_number', 'image_flight', 'folder_sortie', 'image_sortie',
            'raw_image_date', 'folder_date'
        ]]
        
        # Add discrepancy column with normalized dates
        date_df['discrepancy'] = date_df.apply(
            lambda x: f"Folder Date: {x['normalized_folder_date']} ≠ Image Date: {x['normalized_image_date']}", 
            axis=1
        )
        
        # Rename columns for clarity
        date_df = date_df.rename(columns={
            'normalized_folder_date': 'folder_date_normalized',
            'normalized_image_date': 'image_date_normalized'
        })
    else:
        date_df = pd.DataFrame(columns=[
            'folder_name', 'folder_date_normalized', 'image_name', 'image_date_normalized',
            'image_number', 'image_flight', 'folder_sortie', 'image_sortie',
            'raw_image_date', 'folder_date', 'discrepancy'
        ])
    
    # Create Flight mismatch DataFrame
    if mismatched_flights:
        flight_df = pd.DataFrame(mismatched_flights)
        flight_df = flight_df[[
            'folder_name', 'folder_flight', 'image_name', 'image_flight',
            'image_number', 'folder_sortie', 'image_sortie',
            'normalized_folder_date', 'normalized_image_date',
            'normalized_folder_flight'
        ]]
        
        # Add discrepancy column
        flight_df['discrepancy'] = flight_df.apply(
            lambda x: f"Folder Flight: {x['folder_flight']} ≠ Image Flight: {x['image_flight']}", 
            axis=1
        )
        
        # Rename columns for clarity
        flight_df = flight_df.rename(columns={
            'normalized_folder_flight': 'folder_flight_normalized'
        })
    else:
        flight_df = pd.DataFrame(columns=[
            'folder_name', 'folder_flight', 'image_name', 'image_flight',
            'image_number', 'folder_sortie', 'image_sortie',
            'normalized_folder_date', 'normalized_image_date',
            'folder_flight_normalized', 'discrepancy'
        ])
    
    return sortie_df, date_df, flight_df

def save_to_excel(sortie_df, date_df, flight_df, filepath):
    """Save all DataFrames to Excel with different sheets"""
    try:
        with pd.ExcelWriter(filepath, engine='xlsxwriter') as writer:
            # Save SORTIE mismatches
            if not sortie_df.empty:
                sortie_df.to_excel(writer, sheet_name='SORTIE_Mismatches', index=False)
            
            # Save Date mismatches
            if not date_df.empty:
                date_df.to_excel(writer, sheet_name='Date_Mismatches', index=False)
            
            # Save Flight mismatches
            if not flight_df.empty:
                flight_df.to_excel(writer, sheet_name='Flight_Mismatches', index=False)
            
            # Get workbook
            workbook = writer.book
            
            # Define formats
            header_format = workbook.add_format({
                'bold': True,
                'text_wrap': True,
                'valign': 'top',
                'fg_color': '#D7E4BD',
                'border': 1
            })
            
            mismatch_format = workbook.add_format({
                'fg_color': '#FFC7CE',
                'font_color': '#9C0006'
            })
            
            # Apply formatting to each sheet
            for sheet_name, df in [('SORTIE_Mismatches', sortie_df), 
                                    ('Date_Mismatches', date_df),
                                    ('Flight_Mismatches', flight_df)]:
                if sheet_name in writer.sheets and not df.empty:
                    ws = writer.sheets[sheet_name]
                    for col_num, value in enumerate(df.columns.values):
                        ws.write(0, col_num, value, header_format)
                    
                    # Adjust column widths
                    for i, col in enumerate(df.columns):
                        column_width = max(df[col].astype(str).map(len).max(), len(col)) + 2
                        ws.set_column(i, i, min(column_width, 30))
            
            # Add summary sheet
            summary_data = {
                'Category': ['SORTIE Mismatches', 'Date Mismatches', 'Flight Mismatches', 'Total Issues'],
                'Count': [len(sortie_df), len(date_df), len(flight_df), 
                         len(sortie_df) + len(date_df) + len(flight_df)],
                'Timestamp': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 4
            }
            
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            summary_ws = writer.sheets['Summary']
            
            for col_num, value in enumerate(summary_df.columns.values):
                summary_ws.write(0, col_num, value, header_format)
            
            for i, col in enumerate(summary_df.columns):
                column_width = max(summary_df[col].astype(str).map(len).max(), len(col)) + 2
                summary_ws.set_column(i, i, min(column_width, 25))
            
        print(f"\nExcel file saved successfully: {filepath}")
        return True
        
    except Exception as e:
        print(f"Error saving Excel file: {e}")
        return False

def display_preview(sortie_df, date_df, flight_df):
    """Display preview of the results"""
    print("\n" + "=" * 80)
    print("PREVIEW OF RESULTS")
    print("=" * 80)
    
    if not sortie_df.empty:
        print("\n1. SORTIE MISMATCHES (First 5 entries):")
        print("-" * 80)
        preview_cols = ['folder_name', 'folder_sortie', 'image_name', 'image_sortie', 'discrepancy']
        print(sortie_df[preview_cols].head(5).to_string(index=False))
        print(f"\nTotal SORTIE mismatches: {len(sortie_df)}")
        
        # Show date formats found
        print("\nDate formats found in SORTIE mismatches:")
        unique_folder_dates = sortie_df['normalized_folder_date'].unique()[:5]
        unique_image_dates = sortie_df['normalized_image_date'].unique()[:5]
        print(f"  Folder dates: {', '.join([str(d) for d in unique_folder_dates if d])}")
        print(f"  Image dates: {', '.join([str(d) for d in unique_image_dates if d])}")
    else:
        print("\n1. SORTIE MISMATCHES: No mismatches found!")
    
    if not date_df.empty:
        print("\n\n2. DATE MISMATCHES (First 5 entries):")
        print("-" * 80)
        preview_cols = ['folder_name', 'folder_date_normalized', 'image_name', 'image_date_normalized', 'discrepancy']
        print(date_df[preview_cols].head(5).to_string(index=False))
        print(f"\nTotal Date mismatches: {len(date_df)}")
        
        # Show raw date examples
        print("\nRaw date examples from mismatches:")
        for _, row in date_df.head(3).iterrows():
            print(f"  Folder: {row['folder_name']}")
            print(f"    Folder raw: '{row.get('folder_date', '')}', normalized: {row['folder_date_normalized']}")
            print(f"    Image raw: '{row.get('raw_image_date', '')}', normalized: {row['image_date_normalized']}")
            print()
    else:
        print("\n2. DATE MISMATCHES: No mismatches found!")
    
    if not flight_df.empty:
        print("\n\n3. FLIGHT MISMATCHES (First 5 entries):")
        print("-" * 80)
        preview_cols = ['folder_name', 'folder_flight', 'image_name', 'image_flight', 'discrepancy']
        print(flight_df[preview_cols].head(5).to_string(index=False))
        print(f"\nTotal Flight mismatches: {len(flight_df)}")
        
        # Show flight number examples with normalization
        print("\nFlight number normalization examples:")
        for _, row in flight_df.head(3).iterrows():
            print(f"  Folder: '{row['folder_flight']}' -> normalized: '{row['folder_flight_normalized']}'")
            print(f"  Image:  '{row['image_flight']}'")
            print()
    else:
        print("\n3. FLIGHT MISMATCHES: No mismatches found!")
    
    print("\n" + "=" * 80)
    print("STATISTICS")
    print("=" * 80)
    print(f"Total SORTIE mismatches: {len(sortie_df)}")
    print(f"Total Date mismatches: {len(date_df)}")
    print(f"Total Flight mismatches: {len(flight_df)}")
    print(f"Total issues found: {len(sortie_df) + len(date_df) + len(flight_df)}")

def test_date_extraction():
    """Test function to verify date extraction logic"""
    print("\nTesting date extraction with various formats...")
    test_dates = [
        "07_04_1940",
        "1940_04_07",
        "1940-04-07",
        "07-04-1940",
        "19400407",
        "07041940",
        "7_4_1940",
        "1940_4_7",
    ]
    
    for test_date in test_dates:
        year, month, day = extract_date_components(test_date)
        normalized = normalize_date(test_date)
        print(f"  '{test_date}' -> Year: {year}, Month: {month}, Day: {day}, Normalized: {normalized}")

def test_flight_normalization():
    """Test function to verify flight number normalization logic"""
    print("\nTesting flight number normalization...")
    test_flights = [
        "123",
        "0123",
        "00123",
        " 123 ",
        "1-2-3",
        "ABC123",
        "abc123",
        "123A",
        "0123A",
    ]
    
    for flight in test_flights:
        normalized = normalize_flight_number(flight)
        print(f"  '{flight}' -> normalized: '{normalized}'")

def main():
    """Main execution function"""
    print("=" * 80)
    print("SORTIE, DATE, AND FLIGHT VALIDATION TOOL")
    print("=" * 80)
    print(f"Source Path: {folders_path}")
    print(f"Output Path: {output_path}")
    print(f"Output File: {output_filename}")
    print("=" * 80)
    
    # Test date extraction logic
    test_date_extraction()
    
    # Test flight normalization logic
    test_flight_normalization()
    
    # Process all folders and images
    mismatched_sorties, mismatched_dates, mismatched_flights = process_all_folders()
    
    # Create DataFrames
    sortie_df, date_df, flight_df = create_dataframes(mismatched_sorties, mismatched_dates, mismatched_flights)
    
    # Display preview
    display_preview(sortie_df, date_df, flight_df)
    
    # Save to Excel
    print("\n" + "=" * 80)
    print("SAVING TO EXCEL FILE")
    print("=" * 80)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    
    success = save_to_excel(sortie_df, date_df, flight_df, output_filepath)
    
    if success:
        # Display file info
        file_size = os.path.getsize(output_filepath) / 1024  # Convert to KB
        print(f"\n✓ File created successfully!")
        print(f"  Filename: {output_filename}")
        print(f"  Location: {output_path}")
        print(f"  File size: {file_size:.2f} KB")
        
        # Show sheet information
        sheets = []
        if not sortie_df.empty:
            sheets.append(f"SORTIE_Mismatches ({len(sortie_df)} rows)")
        if not date_df.empty:
            sheets.append(f"Date_Mismatches ({len(date_df)} rows)")
        if not flight_df.empty:
            sheets.append(f"Flight_Mismatches ({len(flight_df)} rows)")
        sheets.append("Summary")
        
        print(f"  Worksheets: {', '.join(sheets)}")
    
    print("\n" + "=" * 80)
    print("PROCESS COMPLETED")
    print("=" * 80)

# Execute the main function
if __name__ == "__main__":
    main()

SORTIE, DATE, AND FLIGHT VALIDATION TOOL
Source Path: W:
Output Path: G:\0 BIPLOV\TASKS\FELIX
Output File: 20260930_140606_SORTIE_Date_Flight_Validation.xlsx

Testing date extraction with various formats...
  '07_04_1940' -> Year: 1940, Month: 04, Day: 07, Normalized: 1940-04-07
  '1940_04_07' -> Year: 1940, Month: 04, Day: 07, Normalized: 1940-04-07
  '1940-04-07' -> Year: 1940, Month: 04, Day: 07, Normalized: 1940-04-07
  '07-04-1940' -> Year: 1940, Month: 04, Day: 07, Normalized: 1940-04-07
  '19400407' -> Year: 1940, Month: 04, Day: 07, Normalized: 1940-04-07
  '07041940' -> Year: 0704, Month: 19, Day: 40, Normalized: 0704-19-40
  '7_4_1940' -> Year: 1940, Month: 4, Day: 7, Normalized: 1940-04-07
  '1940_4_7' -> Year: 1940, Month: 4, Day: 7, Normalized: 1940-04-07

Testing flight number normalization...
  '123' -> normalized: '123'
  '0123' -> normalized: '123'
  '00123' -> normalized: '123'
  ' 123 ' -> normalized: '123'
  '1-2-3' -> normalized: '123'
  'ABC123' -> normalized: 'AB

### 2. Finding only the Folders with duplicate SORTIEs

In [1]:
import os
import pandas as pd
from datetime import datetime
from collections import defaultdict

# Path configuration
folders_path = r"W:"
output_path = r"G:\0 BIPLOV\TASKS\FELIX"

# Get current date for filename
current_date = datetime.now().strftime("%Y%m%d")

# Create output filename
output_filename = f"{current_date}_SameSORTIEFolders.xlsx"
output_filepath = os.path.join(output_path, output_filename)

def extract_sortie_info(folder_name):
    """Extract SORTIE and other information from folder name"""
    if not folder_name or '#' not in folder_name:
        return None
    
    parts = folder_name.split('#')
    
    if len(parts) >= 3:
        return {
            'folder_name': folder_name,
            'date': parts[0],
            'flight_number': parts[1],
            'sortie': parts[2],
            'full_path': os.path.join(folders_path, folder_name)
        }
    return None

def process_folders():
    """Main function to process folders and group by SORTIE"""
    # Dictionary to store folders grouped by SORTIE
    sortie_groups = defaultdict(list)
    
    # Get all folders from the path
    try:
        all_items = os.listdir(folders_path)
        folders = [item for item in all_items if os.path.isdir(os.path.join(folders_path, item))]
        
        print(f"Found {len(folders)} folders in {folders_path}")
        
        # Process each folder
        for folder in folders:
            folder_info = extract_sortie_info(folder)
            if folder_info:
                sortie_groups[folder_info['sortie']].append(folder_info)
        
        # Filter to only keep SORTIEs with more than one folder
        duplicate_sorties = {sortie: folders for sortie, folders in sortie_groups.items() 
                           if len(folders) > 1}
        
        print(f"\nFound {len(duplicate_sorties)} SORTIEs with multiple folders")
        
        return duplicate_sorties
        
    except FileNotFoundError:
        print(f"Error: Path {folders_path} not found!")
        return {}
    except PermissionError:
        print(f"Error: Permission denied for path {folders_path}!")
        return {}

def create_dataframe(duplicate_sorties):
    """Create DataFrame from duplicate SORTIEs data"""
    data = []
    
    for sortie, folders in duplicate_sorties.items():
        for folder_info in folders:
            data.append({
                'SORTIE': sortie,
                'Folder_Name': folder_info['folder_name'],
                'Date': folder_info['date'],
                'Flight_Number': folder_info['flight_number'],
                'Full_Path': folder_info['full_path']
            })
    
    df = pd.DataFrame(data)
    
    # Sort by SORTIE and then by Date
    df = df.sort_values(['SORTIE', 'Date'])
    
    return df

def save_to_excel(df, filepath):
    """Save DataFrame to Excel with formatting"""
    try:
        # Create a Pandas Excel writer using XlsxWriter as the engine
        with pd.ExcelWriter(filepath, engine='xlsxwriter') as writer:
            # Write the DataFrame to Excel
            df.to_excel(writer, sheet_name='SameSORTIEFolders', index=False)
            
            # Get the xlsxwriter workbook and worksheet objects
            workbook = writer.book
            worksheet = writer.sheets['SameSORTIEFolders']
            
            # Add some basic formatting
            header_format = workbook.add_format({
                'bold': True,
                'text_wrap': True,
                'valign': 'top',
                'fg_color': '#D7E4BD',
                'border': 1
            })
            
            # Write the column headers with the defined format
            for col_num, value in enumerate(df.columns.values):
                worksheet.write(0, col_num, value, header_format)
            
            # Adjust column widths
            for i, col in enumerate(df.columns):
                column_width = max(df[col].astype(str).map(len).max(), len(col)) + 2
                worksheet.set_column(i, i, min(column_width, 50))
            
        print(f"\nExcel file saved successfully: {filepath}")
        return True
        
    except Exception as e:
        print(f"Error saving Excel file: {e}")
        return False

def main():
    """Main execution function"""
    print("=" * 60)
    print("PROCESSING FOLDERS TO FIND SAME SORTIE")
    print("=" * 60)
    
    # Process folders
    duplicate_sorties = process_folders()
    
    if not duplicate_sorties:
        print("\nNo duplicate SORTIEs found!")
        return
    
    # Create DataFrame
    df = create_dataframe(duplicate_sorties)
    
    # Display table in notebook
    print("\n" + "=" * 60)
    print("FOLDERS GROUPED BY SORTIE (showing first 20 rows)")
    print("=" * 60)
    print(df.head(20).to_string())
    
    print(f"\nTotal rows in table: {len(df)}")
    
    # Show summary statistics
    print("\n" + "=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)
    for sortie, folders in duplicate_sorties.items():
        print(f"SORTIE: {sortie} - {len(folders)} folders")
    
    # Save to Excel
    print("\n" + "=" * 60)
    print("SAVING TO EXCEL FILE")
    print("=" * 60)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    
    success = save_to_excel(df, output_filepath)
    
    if success:
        # Display file info
        file_size = os.path.getsize(output_filepath) / 1024  # Convert to KB
        print(f"\nFile created: {output_filename}")
        print(f"Location: {output_path}")
        print(f"File size: {file_size:.2f} KB")
        print(f"Total SORTIEs: {len(duplicate_sorties)}")
        print(f"Total folders: {len(df)}")

# Execute the main function
if __name__ == "__main__":
    main()

PROCESSING FOLDERS TO FIND SAME SORTIE
Found 3283 folders in W:

Found 2 SORTIEs with multiple folders

FOLDERS GROUPED BY SORTIE (showing first 20 rows)
      SORTIE                 Folder_Name        Date Flight_Number                     Full_Path
2  US32_0885  1945_03_18#00733#US32_0885  1945_03_18         00733  W:1945_03_18#00733#US32_0885
3  US32_0885      1945_04_16#_#US32_0885  1945_04_16             _      W:1945_04_16#_#US32_0885
0          _          1945_03_02#00342#_  1945_03_02         00342          W:1945_03_02#00342#_
1          _          1945_04_20#00466#_  1945_04_20         00466          W:1945_04_20#00466#_

Total rows in table: 4

SUMMARY STATISTICS
SORTIE: _ - 2 folders
SORTIE: US32_0885 - 2 folders

SAVING TO EXCEL FILE

Excel file saved successfully: G:\0 BIPLOV\TASKS\FELIX\20260921_SameSORTIEFolders.xlsx

File created: 20260921_SameSORTIEFolders.xlsx
Location: G:\0 BIPLOV\TASKS\FELIX
File size: 5.68 KB
Total SORTIEs: 2
Total folders: 4


### Check for SORTIE with multiple dates in the database

In [2]:
import psycopg2
import pandas as pd
from datetime import datetime
import os

# Database configuration
DB_CONFIG = {
    'host': '192.168.190.10',
    'port': 5432,
    'database': 'Server_LBA',
    'user': 'LBA_team',  # TO BE FILLED
    'password': 'lba',  # TO BE FILLED
    'sslmode': 'prefer'
}

# Output path
output_dir = r'G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\BIPLOV - FindingMissingDatabaseEntries'

# Create timestamped filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_dir, f'DatabaseSORTIEDateCheck_{timestamp}.xlsx')

# SQL query to find sortie_ori with multiple distinct dates
query = """
SELECT 
    sortie_ori,
    COUNT(DISTINCT "Datum") AS distinct_date_count,
    ARRAY_AGG(DISTINCT "Datum" ORDER BY "Datum") AS dates_found
FROM bildumrahmungen_aktuell
GROUP BY sortie_ori
HAVING COUNT(DISTINCT "Datum") > 1
ORDER BY sortie_ori;
"""

def main():
    try:
        # Connect to database
        conn = psycopg2.connect(**DB_CONFIG)

        # Run query
        df = pd.read_sql_query(query, conn)

        # Save to Excel
        df.to_excel(output_file, index=False)

        print(f"Excel file created: {output_file}")

    except Exception as e:
        print("Error:", e)

    finally:
        if 'conn' in locals():
            conn.close()

if __name__ == "__main__":
    main()

C:\Users\Biplov.Parajuli\AppData\Local\Temp\ipykernel_15204\4147246102.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Excel file created: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\BIPLOV - FindingMissingDatabaseEntries\DatabaseSORTIEDateCheck_20260921_141832.xlsx
